In [0]:
%pip install arch

In [0]:
import numpy as np
import pandas as pd
from arch import arch_model
from pyspark.sql import functions as F

g = (spark.table("workspace.vol_project.gold_btc_daily_rv")
     .select("date", "rv", "daily_ret").toPandas())
g["date"] = pd.to_datetime(g["date"])
g = g.sort_values("date").set_index("date")

r = g["daily_ret"] * 100   # returns in percent

In [0]:
WINDOW = 730   # rolling 2 years, matching your best HAR
DIST   = "t"   # Student-t errors
origins = g.index[(g.index >= "2024-01-01") & (g.index < g.index.max())]
print(len(origins))   # should be 1003

In [0]:
def garch_walk_forward(o):
    preds = {}
    for t in origins:
        train = r[r.index <= t].iloc[-WINDOW:]
        am = arch_model(train, mean="Zero", vol="GARCH", p=1, o=o, q=1, dist=DIST)
        res = am.fit(disp="off")
        preds[t] = res.forecast(horizon=1, reindex=False).variance.iloc[-1, 0] / 10000
    return pd.Series(preds)

garch = garch_walk_forward(o=0)   # GARCH(1,1)
gjr   = garch_walk_forward(o=1)   # GJR-GARCH: lets negative shocks matter more

In [0]:
actual = g["rv"].shift(-1).loc[origins]

def qlike(a, f):
    x = a / f
    return (x - np.log(x) - 1).mean()

print("GARCH QLIKE:", qlike(actual, garch))
print("GJR   QLIKE:", qlike(actual, gjr))
print("HAR rolling (benchmark): 0.2700")

# level check: does GARCH target the same thing as RV?
print("mean squared daily return:", ((g["daily_ret"])**2).mean())
print("mean RV:", g["rv"].mean())

In [0]:
fc = pd.concat([
    pd.DataFrame({"model": "garch_t_730", "forecast": garch}),
    pd.DataFrame({"model": "gjr_t_730",   "forecast": gjr}),
])
fc["actual"] = actual.reindex(fc.index).values
fc.index.name = "origin_date"
fc = fc.reset_index()
fc["target_date"] = (fc["origin_date"] + pd.Timedelta(days=1)).dt.date
fc = fc[["target_date", "model", "forecast", "actual"]]

model_list = ", ".join(f"'{m}'" for m in fc["model"].unique())

(spark.createDataFrame(fc)
    .withColumn("created_at", F.current_timestamp())
    .select("target_date", "model", "forecast", "actual", "created_at")
    .write.format("delta").mode("overwrite")
    .option("replaceWhere", f"model IN ({model_list})")
    .saveAsTable("workspace.vol_project.forecasts"))

In [0]:
display(spark.sql("""
SELECT model,
       COUNT(*)          AS n,
       MIN(target_date)  AS first_day,
       MAX(target_date)  AS last_day,
       AVG(actual / forecast - LN(actual / forecast) - 1) AS qlike
FROM workspace.vol_project.forecasts
GROUP BY model
ORDER BY qlike
"""))